# 🤖 iTech Academy — AI Workshop
## Level 1 (Basic): Retrieval-Augmented Generation (RAG)

---

> **Welcome!** In this notebook you will build a real AI system that can answer questions from *any document you give it* — using 100% free tools.

---

### 📋 What We Will Build Today

```
Your Document (PDF / Text)
        ↓
   Split into Chunks
        ↓
   Turn into Embeddings (numbers)
        ↓
   Store in Vector Database
        ↓
   You Ask a Question
        ↓
   Find Relevant Chunks
        ↓
   LLM Reads Chunks + Answers 🎉
```

### 🛠️ Tools We Use (All FREE)
| Tool | Purpose | Cost |
|------|---------|------|
| **Groq API** | Free LLM (Llama 3.1) | FREE |
| **sentence-transformers** | Convert text to numbers (embeddings) | FREE |
| **ChromaDB** | Store & search embeddings | FREE |
| **LangChain** | Glue everything together | FREE |


---
## 🔧 STEP 0 — Install Libraries

Run this cell **once**. It may take 2–3 minutes the first time.

In [1]:
# Install all required libraries (run once)
!pip install -q groq langchain langchain-community chromadb sentence-transformers pypdf

---
## 🔑 STEP 1 — Set Your FREE Groq API Key

### 📌 How to get your FREE Groq API Key:
1. Go to 👉 **https://console.groq.com**
2. Sign up (free, no credit card)
3. Click **API Keys** → **Create API Key**
4. Copy and paste it below

> ⚠️ **Never share your API key with anyone!**

In [ ]:
import os

# 👇 Paste your Groq API key here


print("✅ API key is set!")

✅ API key is set!


---
## 🧠 SECTION 1 — What is RAG? (Theory + Demo)

### ❓ The Problem with Normal LLMs

Imagine asking ChatGPT:
- *"What is in Chapter 3 of my textbook?"*  ← It has **no idea!**
- *"What did our company announce last week?"*  ← It has **no idea!**
- *"Summarise this document I uploaded?"*  ← It might **hallucinate!**

### 💡 The RAG Solution

**RAG = Give the LLM the right information BEFORE it answers.**

Instead of:
> User: "What is in Chapter 3?"
> LLM: *makes something up* 😬

With RAG:
> User: "What is in Chapter 3?"
> RAG System: *finds Chapter 3 text from your book* 📄
> LLM: *reads the actual text and answers correctly* ✅

---
### 🎯 DEMO 1 — Without RAG (LLM guessing)

In [ ]:
from groq import Groq
GROK_API_KEY = ""

# Connect to Groq (free LLM)
client = Groq(api_key = GROK_API_KEY)

# Ask a question about something the LLM does NOT know
question = "What are the 3 main rules of iTech Academy students?"

response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {"role": "user", "content": question}
    ]
)

print("❌ WITHOUT RAG — LLM is just guessing:")
print("─" * 50)
print(response.choices[0].message.content)

❌ WITHOUT RAG — LLM is just guessing:
──────────────────────────────────────────────────
I couldn't find information on the '3 main rules of iTech Academy students.' iTech Academy seems to be a private school for students pursuing a career in technology, and details about their rules are not widely available.


In [4]:
# Now let's give the LLM the ACTUAL information (this is the RAG idea!)

# Pretend this came from a document
context = """
iTech Academy Student Rules:
1. Always be on time for all sessions and workshops.
2. Respect all instructors, mentors, and fellow students at all times.
3. Submit all assignments before the deadline — no late submissions accepted.
"""

response_with_context = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {
            "role": "user",
            "content": f"Use this information to answer:\n{context}\n\nQuestion: {question}"
        }
    ]
)

print("✅ WITH RAG — LLM reads the actual document:")
print("─" * 50)
print(response_with_context.choices[0].message.content)

✅ WITH RAG — LLM reads the actual document:
──────────────────────────────────────────────────
The 3 main rules of iTech Academy students are:

1. Be on time for all sessions and workshops.
2. Respect all instructors, mentors, and fellow students at all times.
3. Submit all assignments before the deadline.


---
## 📐 SECTION 2 — Embeddings: Turning Text into Numbers

### 🤔 What is an Embedding?

An **embedding** is a list of numbers (a vector) that represents the *meaning* of text.

```
"dog"   → [0.2,  0.8, -0.1,  0.5, ...] (384 numbers)
"puppy" → [0.19, 0.78, -0.09, 0.48, ...]  ← very similar numbers!
"car"   → [0.9, -0.2,  0.7, -0.3, ...]   ← very different numbers!
```

**Why?** Because computers can't compare meaning — but they can compare numbers! 🔢

---
### 🎯 DEMO 2 — See Embeddings in Action

In [5]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load a free embedding model (downloads once, ~90MB)
print("⏳ Loading embedding model... (first time takes ~30 seconds)")
model = SentenceTransformer("all-MiniLM-L6-v2")
print("✅ Embedding model ready!")

/Users/syedaskari/itech-workshop/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


⏳ Loading embedding model... (first time takes ~30 seconds)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14434.12it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedding model ready!


In [6]:
# Let's embed some sentences and see what they look like
sentences = [
    "I love programming in Python",
    "Python is my favourite coding language",   # similar to above
    "The weather today is sunny and warm",        # very different
]

embeddings = model.encode(sentences)

print(f"Each sentence becomes {len(embeddings[0])} numbers!\n")
print("First 8 numbers of each embedding:")
for i, (s, e) in enumerate(zip(sentences, embeddings)):
    print(f"  Sentence {i+1}: {e[:8].round(3)}")
    print(f"           ↑ '{s[:40]}...'")
    print()

Each sentence becomes 384 numbers!

First 8 numbers of each embedding:
  Sentence 1: [-0.057  0.013 -0.034  0.024 -0.017 -0.169  0.037  0.08 ]
           ↑ 'I love programming in Python...'

  Sentence 2: [-0.088 -0.025 -0.026  0.018 -0.008 -0.125  0.036  0.046]
           ↑ 'Python is my favourite coding language...'

  Sentence 3: [-0.009  0.113  0.117  0.07   0.056 -0.029  0.089 -0.096]
           ↑ 'The weather today is sunny and warm...'



In [7]:
from sklearn.metrics.pairwise import cosine_similarity

# Calculate similarity between sentences (1.0 = identical, 0.0 = completely different)
sim_matrix = cosine_similarity(embeddings)

print("🔍 Similarity Scores (how similar are the sentences?):")
print("─" * 55)
pairs = [
    (0, 1, "Python sentence 1  vs  Python sentence 2"),
    (0, 2, "Python sentence 1  vs  Weather sentence "),
    (1, 2, "Python sentence 2  vs  Weather sentence "),
]
for i, j, label in pairs:
    score = sim_matrix[i][j]
    bar   = "█" * int(score * 20)
    print(f"{label}")
    print(f"  Score: {score:.3f}  |{bar}|")
    print()

print("\n💡 Notice: similar meaning = high score, different meaning = low score!")

🔍 Similarity Scores (how similar are the sentences?):
───────────────────────────────────────────────────────
Python sentence 1  vs  Python sentence 2
  Score: 0.852  |█████████████████|

Python sentence 1  vs  Weather sentence 
  Score: 0.042  ||

Python sentence 2  vs  Weather sentence 
  Score: 0.047  ||


💡 Notice: similar meaning = high score, different meaning = low score!


---
## 🗄️ SECTION 3 — Vector Database (ChromaDB)

### 🤔 What is a Vector Database?

A normal database stores text and lets you search by **exact words**.

A **vector database** stores embeddings (numbers) and lets you search by **meaning**.

```
Normal DB search:  "python"  → finds only documents with the word "python"
Vector DB search:  "python"  → finds documents about python, coding, programming,
                                 scripting... even if they don't say "python"!
```

---
### 🎯 DEMO 3 — Store and Search with ChromaDB

In [ ]:
import chromadb

# Create a local database (saves to your computer, no internet needed)
chroma_client = chromadb.Client()

# Create a collection (like a table in a normal database)
collection = chroma_client.create_collection(name="demo_facts")

# Our "documents" — pretend these are from a textbook
facts = [
    "Python was created by Guido van Rossum in 1991.",
    "Machine learning is a subset of artificial intelligence.",
    "Neural networks are inspired by the human brain.",
    "ChromaDB is an open-source vector database.",
    "The Eiffel Tower is located in Paris, France.",
    "Deep learning uses multiple layers of neural networks.",
    "Python is one of the most popular programming languages in 2024.",
]

# Embed all facts and store them
fact_embeddings = model.encode(facts).tolist()

collection.add(
    documents=facts,
    i=fact_embeddings,
    ids=[f"fact_{i}" for i in range(len(facts))]
)

print(f"✅ Stored {len(facts)} facts in ChromaDB!")

✅ Stored 7 facts in ChromaDB!


In [9]:
# Now let's SEARCH by meaning!

def semantic_search(query, top_k=3):
    """Search the database by meaning, not exact words."""
    query_embedding = model.encode([query]).tolist()
    
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k
    )
    
    print(f"🔍 Query: '{query}'")
    print("─" * 50)
    print("Top results (most relevant first):")
    for i, (doc, dist) in enumerate(zip(
        results["documents"][0],
        results["distances"][0]
    )):
        relevance = 1 - dist  # convert distance to similarity
        print(f"  {i+1}. [{relevance:.2f}] {doc}")
    print()

# Try searching — notice it finds related content even with different words!
semantic_search("Who invented Python?")
semantic_search("How does AI relate to the brain?")
semantic_search("Where can I see the Eiffel Tower?")

🔍 Query: 'Who invented Python?'
──────────────────────────────────────────────────
Top results (most relevant first):
  1. [0.43] Python was created by Guido van Rossum in 1991.
  2. [0.16] Python is one of the most popular programming languages in 2024.
  3. [-0.62] Neural networks are inspired by the human brain.

🔍 Query: 'How does AI relate to the brain?'
──────────────────────────────────────────────────
Top results (most relevant first):
  1. [0.25] Neural networks are inspired by the human brain.
  2. [-0.06] Machine learning is a subset of artificial intelligence.
  3. [-0.29] Deep learning uses multiple layers of neural networks.

🔍 Query: 'Where can I see the Eiffel Tower?'
──────────────────────────────────────────────────
Top results (most relevant first):
  1. [0.56] The Eiffel Tower is located in Paris, France.
  2. [-0.94] Deep learning uses multiple layers of neural networks.
  3. [-0.95] ChromaDB is an open-source vector database.



In [10]:
# 🎮 YOUR TURN! Try your own search query below:

my_query = "Tell me about deep learning"  # ← Change this!
semantic_search(my_query)

🔍 Query: 'Tell me about deep learning'
──────────────────────────────────────────────────
Top results (most relevant first):
  1. [0.35] Deep learning uses multiple layers of neural networks.
  2. [-0.11] Machine learning is a subset of artificial intelligence.
  3. [-0.24] Neural networks are inspired by the human brain.



---
## 📄 SECTION 4 — Working with Real Documents

### Loading and Chunking

Real documents are too long to fit in one embedding.
We need to split them into **chunks** first.

```
Long Document (10,000 words)
  ↓  split
Chunk 1 (200 words) | Chunk 2 (200 words) | Chunk 3 (200 words) | ...
  ↓  embed each chunk separately
Vector 1            | Vector 2            | Vector 3            | ...
```

**Why overlap?**  So we don't cut a sentence in half and lose meaning!

---

In [11]:
# Simple chunking function — easy to understand!

def chunk_text(text, chunk_size=200, overlap=40):
    """
    Split text into overlapping chunks.
    
    chunk_size = how many characters per chunk
    overlap    = how many characters to repeat between chunks
                 (so we don't cut sentences at bad places)
    """
    chunks = []
    start  = 0
    
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk.strip())
        start = end - overlap  # move back by overlap amount
    
    return chunks


# Test it with a sample text
sample_text = """
Artificial Intelligence is transforming every industry. 
From healthcare to finance, AI systems are helping humans make better decisions.
Machine learning is a type of AI where computers learn from data without being explicitly programmed.
Deep learning is an advanced form of machine learning that uses neural networks with many layers.
Large Language Models like GPT and Llama are trained on billions of text samples.
These models can understand and generate human-like text with remarkable accuracy.
RAG combines the power of LLMs with real-time information retrieval.
This makes AI systems more accurate and reduces hallucinations significantly.
"""

chunks = chunk_text(sample_text, chunk_size=200, overlap=40)

print(f"Original text: {len(sample_text)} characters")
print(f"Number of chunks: {len(chunks)}")
print()
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1} ({len(chunk)} chars):")
    print(f"  '{chunk[:80]}...'")
    print()

Original text: 651 characters
Number of chunks: 5

Chunk 1 (199 chars):
  'Artificial Intelligence is transforming every industry. 
From healthcare to fina...'

Chunk 2 (199 chars):
  'type of AI where computers learn from data without being explicitly programmed.
...'

Chunk 3 (199 chars):
  'with many layers.
Large Language Models like GPT and Llama are trained on billio...'

Chunk 4 (170 chars):
  'th remarkable accuracy.
RAG combines the power of LLMs with real-time informatio...'

Chunk 5 (10 chars):
  'ificantly....'



In [12]:
# Let's also show how to load from a REAL .txt file
# First we create a sample file (in class you can use your own!)

sample_document = """iTech Academy — AI Workshop Reference Guide

What is Artificial Intelligence?
Artificial Intelligence (AI) refers to computer systems that can perform tasks that normally require human intelligence. These tasks include visual perception, speech recognition, decision-making, and language understanding.

What is Machine Learning?
Machine Learning is a branch of AI that gives computers the ability to learn from data without being explicitly programmed. Instead of writing rules manually, the computer finds patterns in data on its own.

What is a Large Language Model?
A Large Language Model (LLM) is a type of AI trained on massive amounts of text. It learns the patterns of language and can generate, summarise, translate, and answer questions about text. Examples include GPT-4, Llama 3, and Mistral.

What is RAG?
Retrieval-Augmented Generation (RAG) is a technique that combines an LLM with a search system. Instead of relying only on what the model was trained on, RAG retrieves relevant documents from a database and uses them as context when generating answers. This makes answers more accurate and up-to-date.

What are Embeddings?
Embeddings are numerical representations of text. They convert words and sentences into vectors (lists of numbers) that capture the meaning of the text. Similar pieces of text will have similar embeddings, which allows computers to find related content even when the exact words are different.

What is ChromaDB?
ChromaDB is an open-source vector database that stores embeddings. It allows you to save thousands of text embeddings and search through them quickly by meaning. It runs locally on your computer with no internet connection required.

What is Groq?
Groq is a company that provides ultra-fast AI inference. Their free API tier gives developers access to powerful open-source models like Llama 3.1 at no cost. It is perfect for learning and building projects.
"""

# Save to file
with open("itech_guide.txt", "w") as f:
    f.write(sample_document)

# Load from file
with open("itech_guide.txt", "r") as f:
    loaded_text = f.read()

print(f"✅ Loaded document: {len(loaded_text)} characters")
print(f"Preview: {loaded_text[:150]}...")

✅ Loaded document: 1912 characters
Preview: iTech Academy — AI Workshop Reference Guide

What is Artificial Intelligence?
Artificial Intelligence (AI) refers to computer systems that can perform...


---
## 🧪 LAB 1 — Build the Full RAG Pipeline

> **This is what it's all been building towards!** We put everything together.

### The complete pipeline in one place:
```
1. Load document
2. Split into chunks
3. Embed each chunk
4. Store in ChromaDB
5. User asks a question
6. Find top-3 most relevant chunks
7. Send chunks + question to LLM
8. Get a grounded answer! ✅
```
---

In [13]:
# ─────────────────────────────────────────────
#  STEP 1 & 2: Load and Chunk the Document
# ─────────────────────────────────────────────

with open("itech_guide.txt", "r") as f:
    document = f.read()

# Use a larger chunk size for real documents
doc_chunks = chunk_text(document, chunk_size=400, overlap=60)

print(f"✅ Step 1-2 done: {len(doc_chunks)} chunks created")

✅ Step 1-2 done: 6 chunks created


In [14]:
# ─────────────────────────────────────────────
#  STEP 3 & 4: Embed and Store in ChromaDB
# ─────────────────────────────────────────────

import chromadb

# Fresh database
rag_client     = chromadb.Client()
rag_collection = rag_client.create_collection(name="itech_docs")

# Embed all chunks
chunk_embeddings = model.encode(doc_chunks).tolist()

# Store in ChromaDB
rag_collection.add(
    documents=doc_chunks,
    embeddings=chunk_embeddings,
    ids=[f"chunk_{i}" for i in range(len(doc_chunks))]
)

print(f"✅ Step 3-4 done: {len(doc_chunks)} chunks embedded and stored!")

✅ Step 3-4 done: 6 chunks embedded and stored!


In [15]:
# ─────────────────────────────────────────────
#  STEP 5-8: The RAG Q&A Function
# ─────────────────────────────────────────────

def ask_rag(question, top_k=3, show_sources=True):
    """
    Ask a question and get a grounded answer from the document.
    
    Steps inside this function:
      1. Embed the question
      2. Find the top_k most relevant chunks
      3. Build a prompt with the chunks as context
      4. Send to Groq (Llama 3.1) for final answer
    """
    
    # Step 5: Embed the question
    q_embedding = model.encode([question]).tolist()
    
    # Step 6: Find relevant chunks
    results = rag_collection.query(
        query_embeddings=q_embedding,
        n_results=top_k
    )
    relevant_chunks = results["documents"][0]
    
    # Show sources if requested
    if show_sources:
        print("📄 Retrieved chunks used as context:")
        for i, chunk in enumerate(relevant_chunks):
            print(f"  [{i+1}] {chunk[:100]}...")
        print()
    
    # Step 7: Build the prompt
    context = "\n\n".join(relevant_chunks)
    
    prompt = f"""You are a helpful assistant. Use ONLY the context below to answer the question.
If the answer is not in the context, say "I don't have that information in my documents."

Context:
{context}

Question: {question}

Answer:"""
    
    # Step 8: Send to LLM and get answer
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=300
    )
    
    answer = response.choices[0].message.content
    
    print(f"❓ Question: {question}")
    print("─" * 50)
    print(f"✅ Answer: {answer}")
    print()
    return answer


print("✅ RAG function ready! Let's test it below.")

✅ RAG function ready! Let's test it below.


In [16]:
# 🎯 Test the RAG system!

ask_rag("What is RAG and why is it useful?")

📄 Retrieved chunks used as context:
  [1] guage and can generate, summarise, translate, and answer questions about text. Examples include GPT-...
  [2] and uses them as context when generating answers. This makes answers more accurate and up-to-date.

...
  [3] computers to find related content even when the exact words are different.

What is ChromaDB?
Chroma...

❓ Question: What is RAG and why is it useful?
──────────────────────────────────────────────────
✅ Answer: RAG stands for Retrieval-Augmented Generation, a technique that combines an LLM (Large Language Model) with a search system. It retrieves relevant documents from a database and uses them as context when generating answers, making answers more accurate and up-to-date.

This makes RAG useful because it allows the model to rely on the latest information available, rather than just what it was trained on. It also enables the model to access a vast amount of relevant information from a database, which helps to improve the acc

'RAG stands for Retrieval-Augmented Generation, a technique that combines an LLM (Large Language Model) with a search system. It retrieves relevant documents from a database and uses them as context when generating answers, making answers more accurate and up-to-date.\n\nThis makes RAG useful because it allows the model to rely on the latest information available, rather than just what it was trained on. It also enables the model to access a vast amount of relevant information from a database, which helps to improve the accuracy and reliability of its responses.'

In [17]:
ask_rag("What is ChromaDB?")

📄 Retrieved chunks used as context:
  [1] computers to find related content even when the exact words are different.

What is ChromaDB?
Chroma...
  [2] guage and can generate, summarise, translate, and answer questions about text. Examples include GPT-...
  [3] and uses them as context when generating answers. This makes answers more accurate and up-to-date.

...

❓ Question: What is ChromaDB?
──────────────────────────────────────────────────
✅ Answer: ChromaDB is an open-source vector database that stores embeddings, allowing you to save thousands of text embeddings and search through them quickly by meaning. It runs locally on your computer with no internet connection required.



'ChromaDB is an open-source vector database that stores embeddings, allowing you to save thousands of text embeddings and search through them quickly by meaning. It runs locally on your computer with no internet connection required.'

In [18]:
ask_rag("What is the difference between AI and Machine Learning?")

📄 Retrieved chunks used as context:
  [1] iTech Academy — AI Workshop Reference Guide

What is Artificial Intelligence?
Artificial Intelligenc...
  [2] arning is a branch of AI that gives computers the ability to learn from data without being explicitl...
  [3] q?
Groq is a company that provides ultra-fast AI inference. Their free API tier gives developers acc...

❓ Question: What is the difference between AI and Machine Learning?
──────────────────────────────────────────────────
✅ Answer: According to the context, AI refers to computer systems that can perform tasks that normally require human intelligence, and these tasks include visual perception, speech recognition, decision-making, and language understanding.

Machine Learning is a branch of AI that focuses on giving computers the ability to learn from data without being explicitly programmed. 

The main difference between AI and Machine Learning is that AI is a broader field that encompasses various tasks and capabilities, while

'According to the context, AI refers to computer systems that can perform tasks that normally require human intelligence, and these tasks include visual perception, speech recognition, decision-making, and language understanding.\n\nMachine Learning is a branch of AI that focuses on giving computers the ability to learn from data without being explicitly programmed. \n\nThe main difference between AI and Machine Learning is that AI is a broader field that encompasses various tasks and capabilities, while Machine Learning is a specific area within AI that deals with learning from data and finding patterns without manual programming.'

In [19]:
# 🧪 What happens when we ask something NOT in the document?
ask_rag("What is the capital city of India?")

📄 Retrieved chunks used as context:
  [1] iTech Academy — AI Workshop Reference Guide

What is Artificial Intelligence?
Artificial Intelligenc...
  [2] arning is a branch of AI that gives computers the ability to learn from data without being explicitl...
  [3] guage and can generate, summarise, translate, and answer questions about text. Examples include GPT-...

❓ Question: What is the capital city of India?
──────────────────────────────────────────────────
✅ Answer: I don't have that information in my documents.



"I don't have that information in my documents."

---
## 🎮 MINI CHALLENGE — Try Your Own Document!

Replace the `my_document` text below with:
- Your **college notes**
- A **Wikipedia article** you copied
- A **chapter from a textbook**
- Any text you like!

Then ask questions about it. 🚀

In [ ]:
# 👇 Replace this with your own text!
my_document = """
Write or paste your own document here.
It can be any text — notes, articles, anything!
Try to use at least 5-6 sentences so there is enough to search through.
"""

# Build a new RAG system from your document
my_chunks     = chunk_text(my_document, chunk_size=300, overlap=50)
my_embeddings = model.encode(my_chunks).tolist()

my_client     = chromadb.Client()
my_collection = my_client.create_collection(name="my_docs")
my_collection.add(
    documents=my_chunks,
    embeddings=my_embeddings,
    ids=[f"chunk_{i}" for i in range(len(my_chunks))]
)

def ask_my_rag(question):
    q_emb    = model.encode([question]).tolist()
    results  = my_collection.query(query_embeddings=q_emb, n_results=2)
    context  = "\n".join(results["documents"][0])
    prompt   = f"Use this context to answer.\n\nContext: {context}\n\nQuestion: {question}\nAnswer:"
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )
    print(f"❓ {question}")
    print(f"✅ {response.choices[0].message.content}\n")

print(f"✅ Your personal RAG system is ready with {len(my_chunks)} chunks!")
print("Now run the cell below and ask it anything about your document!")

In [ ]:
# 🎮 Ask anything about your own document!
ask_my_rag("Write your question here")  # ← Change this!

---
## 🏁 SECTION 5 — Quick Recap & What's Next

### ✅ What You Built Today

| Step | What You Did | Tool |
|------|-------------|------|
| 1 | Connected to a free LLM | Groq API (Llama 3.1) |
| 2 | Understood embeddings | sentence-transformers |
| 3 | Built semantic search | ChromaDB |
| 4 | Loaded & chunked a document | Python |
| 5 | Built a full RAG Q&A system | All of the above! |

### 🚀 What's Next — Level 2 (AI Agents)

After lunch you will learn:
- How to build an AI **Agent** that can use tools (search, calculator, code)
- How to combine your **RAG system** with an **Agent** for a super-powered AI assistant
- All using the same free tools!

---
### 💡 Key Terms to Remember

| Term | Simple Definition |
|------|-----------------|
| **LLM** | AI that reads and writes text |
| **Embedding** | Text turned into a list of numbers |
| **Vector DB** | Database that stores and searches embeddings |
| **Chunk** | A small piece of a larger document |
| **RAG** | Find relevant text → give to LLM → get accurate answer |
| **Hallucination** | When an LLM makes up facts that aren't real |

---

> 🎉 **Congratulations!** You just built a RAG system from scratch using 100% free tools.
> You can now take this notebook home and build a Q&A system over ANY document you want!

---
*iTech Academy  |  AI Workshop 2025  |  All tools used are 100% free & open source*